Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [1]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

model=qwen3.5:2b


## LangChain Hello World and types of tasks performed with LLMs

### Instal libraries

In [2]:
# before running the notebook, make sure you have Anaconda installed: https://www.anaconda.com/docs/getting-started/anaconda/install
# install required packages (only once)
# (setup cell already installs what this notebook needs)

In [3]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm()

# simple prompt and request to the LLM API
response = llm.invoke("Write a short apology to a customer whose order is late.")

print("Model response:\n")
print(response.content)

Model response:

Subject: Apology for the Delay in Your Order #[Order Number]

Dear [Customer Name],

I am writing to sincerely apologize for the delay with your recent order, which arrived later than expected. We understand how frustrating this can be, and we are sorry for any inconvenience this has caused you.

We have already shipped a replacement item to your address as soon as possible. Please check your email or account dashboard for details on the new shipment. If you need anything else from us today, please don't hesitate to reach out.

Thank you for your patience and understanding. We value your business and are committed to making this right.

Sincerely,

[Your Name]  
[Your Title/Company Name]


### LangChain Message List

In [4]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

llm = make_llm()

system_message = SystemMessage(content="You are a helpful assistant. Always answer in Dutch.")
human_message = HumanMessage(content="Write a short apology for a late delivery.")

messages = [system_message, human_message]

response = llm.invoke(messages)

print("Model response:\n")
print(response.content)

print("\nType of SystemMessage:", type(system_message))
print("Type of HumanMessage: ", type(human_message))
print("Type of response:     ", type(response))

Model response:

Natuurlijk! Hier is een korte en vriendelijke excuses voor de vertraging:

"Bedankt dat je contact hebt opgenomen. Ik ben erg bedroegd dat mijn pakket niet zo snel mogelijk aankomt. Ik zal dit proberen te voorkomen in de toekomst. Bedankt voor je geduld."

Type of SystemMessage: <class 'langchain_core.messages.system.SystemMessage'>
Type of HumanMessage:  <class 'langchain_core.messages.human.HumanMessage'>
Type of response:      <class 'langchain_core.messages.ai.AIMessage'>


### Text generation

In [5]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm(temperature=1)

response = llm.invoke("Generate a recipe for sweet cheesecake with tuna, broccoli and onion.")

print("Model response:\n")
print(response.content)

Model response:

This is essentially your choice! It is not just the standard "New York" style cheesecake you know. Instead, we are building a **Sweet & Savory Truffle-Style Sweet Cheesecake** (often called a "Salad Cake") that incorporates three distinct ingredients:

1.  **Tuna**: Provides high-quality fish for a savory note and protein boost.
2.  **Broccoli**: Adds crunch, fiber, and bright yellow color for contrast.
3.  **Onion**: The secret agent hiding in the creamy base to add depth without overpowering the fruit sweetness.

*Note: This recipe is designed for an individual portion or a small dessert slice, not a family dinner.*

---

### **Sweet Savory "Salad" Cheesecake**
**Serves:** 1–2 (Individual portions)
**Prep Time:** 15 mins | **Bake Time:** 45–50 mins

#### **Ingredients**

*   **Base Layer:**
    *   1 cup whole milk (or heavy cream if you prefer a thicker texture)
    *   2 large eggs + 1 yolk for richness
    *   3 tablespoons granulated sugar
    *   ½ teaspoon vani

### Classification

In [6]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm()

articles = [
    "The government announced new tax reforms today.",
    "The local team won the championship in a thrilling match.",
    "New advancements in AI are reshaping the tech industry.",
    "The art exhibit showcased contemporary works by emerging artists.",
    "New guidelines for a healthy diet were published by the health department."
]
for subject in articles:
    response = llm.invoke("Classify texts into groups Politics, Sport, Technology, Culture, Health. Return only single word with category."
    f"Text: {subject}")

    print(subject)
    print(response.content)

The government announced new tax reforms today.
Politics
The local team won the championship in a thrilling match.
Sport


New advancements in AI are reshaping the tech industry.
Technology


The art exhibit showcased contemporary works by emerging artists.
Culture
New guidelines for a healthy diet were published by the health department.
Health


### Sentiment analysis

In [7]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm()
# Sample reviews
reviews = [
    "This product is amazing! I loved it.",
    "I am very disappointed. The product broke after one use.",
    "It's okay, does the job but nothing special."
]
for review in reviews:
    response = llm.invoke(f"Rate the sentiment of a review. Return a float from 0 to 1 with 0 being unhappy and 1 being very happy: {review}")

    print("Model response:\n")
    print(review)
    print(response.content)

Model response:

This product is amazing! I loved it.
1


Model response:

I am very disappointed. The product broke after one use.
0.95


Model response:

It's okay, does the job but nothing special.
0.4


### Document analysis

In [8]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm()

file = open(data("annual_report.html"), "r")
document = file.read()
response = llm.invoke(f"Analyze the attached document and find information about the company's annual revenue. {document}")
print("Model response:\n")
print(response.content)

Model response:

Based on the provided document table, here is the analysis of the company's annual revenue:

### **Key Financial Data**
The table presents data for two years: **2019** and **2018**. The relevant figures are found in the row labeled **"Umsatzerlöse"** (Revenue), which is measured in **Mio. EUR** (Million Euros).

*   **2019 Revenue:** 55,680 Mio. EUR
*   **2018 Revenue:** 59,248 Mio. EUR

### **Trend Analysis**
*   **Decline:** There was a decrease in revenue from the previous year.
*   **Percentage Change:** The document indicates a change of **-6,0%** (a decline of 6.0%).
*   **Absolute Difference:** The revenue dropped by approximately **3,568 Million Euros** (59,248 - 55,680).

### **Conclusion**
The company's annual revenue decreased from **€59,248 million** in 2018 to **€55,680 million** in 2019. This represents a **6.0% decline** year-over-year.


### Machine translation

In [9]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm()
response = llm.invoke("Translate the text below into Dutch:\n I'm foreigner and I don't speak german fluently.")

print("Model response:\n")
print(response.content)

Model response:

Ik ben een vreemdeling en ik spreken Duits niet vloeiend.


### Question answering

In [10]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm()

response = llm.invoke("Answer the following questions based on the attached text:\n"
"LangChain is a framework for working with large language models.\n"
"Chains in LangChain are data flows between prompts, models, and parsers.\n"
"Retriever allows you to search for information in a vector database.\n"
"Question: What are Chains used for in LangChain?")

print("Model response:\n")
print(response.content)

Model response:

Based on the text provided, **Chains** in LangChain are used as **data flows**. Specifically, they connect three key components: prompts, models, and parsers. This structure allows you to orchestrate a sequence of operations where information moves from an input prompt through a model (likely for generation or analysis) and then to a parser before being processed further.


### Summarization

In [11]:
# import libraries
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# load environment variables from the .env file
load_dotenv()

# create the OpenAI API client with the selected model
llm = make_llm()
with open(data("nad_niemnem.txt"), "r", encoding="utf-8") as file:
    document = file.read()

response = llm.invoke(f"Write a short summary (approx. 500 words) of the attached text.\n{document[:1800]}")

print("Model repsonse:\n")
print(response.content)

Model repsonse:

The provided text is an excerpt from the first chapter of Tom I's novel *Nad Niemnem* (Over the River), written by Eliza Orzeszkowa. The narrative opens with a vivid description of a summer holiday in Poland, characterized by an atmosphere of intense joy and freedom. The author paints a picture of a world where nature is vibrant: fields are lush with green grain, the sky is blue, and the sun shines brightly. This sense of happiness permeates the landscape, which consists of rolling hills with dark bark and forests, as well as a wide, flat plain dominated by a sandy riverbank.

The setting is further enriched by the presence of numerous small manor houses scattered along the river's edge. These dwellings create a rhythmic pattern across the land, their chimneys emitting smoke that rises into the clear air. The windows glow like sparks from the sun, and new straw roofs blend with the blue sky and green foliage. White roads wind through the plain, dotted with patches of g